In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

print("Librerías cargadas correctamente")

Librerías cargadas correctamente


In [2]:
# Cargamos las dos sheets y las unimos
df09 = pd.read_excel('../data/raw/online_retail_II.xlsx', 
                      sheet_name='Year 2009-2010')
df10 = pd.read_excel('../data/raw/online_retail_II.xlsx', 
                      sheet_name='Year 2010-2011')

df = pd.concat([df09, df10], ignore_index=True)

print(f"Sheet 2009-2010: {df09.shape[0]:,} filas")
print(f"Sheet 2010-2011: {df10.shape[0]:,} filas")
print(f"Total combinado: {df.shape[0]:,} filas · {df.shape[1]} columnas")

Sheet 2009-2010: 525,461 filas
Sheet 2010-2011: 541,910 filas
Total combinado: 1,067,371 filas · 8 columnas


In [3]:
# Estructura del dataset
print("=== COLUMNAS Y TIPOS ===")
print(df.dtypes)
print()

# Valores nulos
print("=== VALORES NULOS ===")
nulls = df.isnull().sum()
nulls_pct = (nulls / len(df) * 100).round(2)
null_df = pd.DataFrame({'Nulos': nulls, 'Porcentaje %': nulls_pct})
print(null_df[null_df['Nulos'] > 0])
print()

# Duplicados
print("=== DUPLICADOS ===")
print(f"Filas duplicadas exactas: {df.duplicated().sum():,}")

=== COLUMNAS Y TIPOS ===
Invoice                object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
Price                 float64
Customer ID           float64
Country                   str
dtype: object

=== VALORES NULOS ===
              Nulos  Porcentaje %
Description    4382          0.41
Customer ID  243007         22.77

=== DUPLICADOS ===
Filas duplicadas exactas: 34,335


In [4]:
print("=== RANGOS DE VARIABLES CLAVE ===")
print(f"Quantity  — min: {df['Quantity'].min():,}  max: {df['Quantity'].max():,}")
print(f"Price     — min: {df['Price'].min():.2f}  max: {df['Price'].max():.2f}")
print()

print("=== TRANSACCIONES CON QUANTITY NEGATIVA (devoluciones) ===")
returns = df[df['Quantity'] < 0]
print(f"Filas: {len(returns):,} ({len(returns)/len(df)*100:.1f}% del total)")
print()

print("=== TRANSACCIONES CON PRICE = 0 ===")
zero_price = df[df['Price'] == 0]
print(f"Filas: {len(zero_price):,}")
print()

print("=== STOCK CODES ANÓMALOS ===")
anomalos = ['POST', 'D', 'M', 'BANK CHARGES', 'AMAZONFEE', 'DOT', 'CRUK']
print(df[df['StockCode'].isin(anomalos)]['StockCode'].value_counts())

=== RANGOS DE VARIABLES CLAVE ===
Quantity  — min: -80,995  max: 80,995
Price     — min: -53594.36  max: 38970.00

=== TRANSACCIONES CON QUANTITY NEGATIVA (devoluciones) ===
Filas: 22,950 (2.2% del total)

=== TRANSACCIONES CON PRICE = 0 ===
Filas: 6,202

=== STOCK CODES ANÓMALOS ===
StockCode
POST            2122
DOT             1446
M               1421
D                177
BANK CHARGES     102
AMAZONFEE         43
CRUK              16
Name: count, dtype: int64


In [9]:
# Separamos devoluciones antes de eliminarlas — son datos valiosos
df_returns = df[df['Quantity'] < 0].copy()
df_returns['is_return'] = True

# Dataset limpio
df_clean = df[
    (df['Quantity'] > 0) &
    (df['Price'] > 0) &
    (df['Customer ID'].notna()) &
    (~df['StockCode'].isin(['POST', 'D', 'M', 'BANK CHARGES', 
                             'AMAZONFEE', 'DOT', 'CRUK']))
].copy()

# Revenue por línea
df_clean['Revenue'] = df_clean['Quantity'] * df_clean['Price']

# Aseguramos tipo datetime
df_clean['InvoiceDate'] = pd.to_datetime(df_clean['InvoiceDate'])

# Limpiar tipos mixtos antes de guardar
df_clean['StockCode'] = df_clean['StockCode'].astype(str)
df_clean['Customer ID'] = df_clean['Customer ID'].astype(str)
df_returns['StockCode'] = df_returns['StockCode'].astype(str)
df_returns['Customer ID'] = df_returns['Customer ID'].astype(str)

print("=== RESULTADO DE LA LIMPIEZA ===")
print(f"Filas originales:      {len(df):,}")
print(f"Filas limpias:         {len(df_clean):,}")
print(f"Filas eliminadas:      {len(df) - len(df_clean):,} ({(1 - len(df_clean)/len(df))*100:.1f}%)")
print(f"Devoluciones guardadas: {len(df_returns):,}")
print(f"Clientes únicos:       {df_clean['Customer ID'].nunique():,}")
print(f"Periodo:               {df_clean['InvoiceDate'].min().date()} → {df_clean['InvoiceDate'].max().date()}")

=== RESULTADO DE LA LIMPIEZA ===
Filas originales:      1,067,371
Filas limpias:         802,949
Filas eliminadas:      264,422 (24.8%)
Devoluciones guardadas: 22,950
Clientes únicos:       5,862
Periodo:               2009-12-01 → 2011-12-09


In [11]:
# Limpiar todos los tipos mixtos en ambos dataframes
for col in df_clean.select_dtypes(include='object').columns:
    df_clean[col] = df_clean[col].astype(str)

for col in df_returns.select_dtypes(include='object').columns:
    df_returns[col] = df_returns[col].astype(str)

# Guardar
df_clean.to_parquet('../data/processed/transactions_clean.parquet', index=False)
df_returns.to_parquet('../data/processed/returns.parquet', index=False)

print("Archivos guardados en data/processed/")
print(f"  transactions_clean.parquet  ({len(df_clean):,} filas)")
print(f"  returns.parquet             ({len(df_returns):,} filas)")

Archivos guardados en data/processed/
  transactions_clean.parquet  (802,949 filas)
  returns.parquet             (22,950 filas)
